# Verify Fully Connected MNIST Classifier

This notebook demonstrates formal verification of the MNIST FC classifier using NNV-Python.

**Verification Task**: Local robustness against adversarial perturbations

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import time

# Import NNV-Python
from nnv_py.sets import Star
from nnv_py.nn.reach.reach_star import reach_star_exact

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')  # Verification on CPU
print("NNV-Python loaded successfully!")

## 2. Load Trained Model

In [ ]:
# Define model architecture (same as training)
class FCMnistClassifier(nn.Module):
    def __init__(self):
        super(FCMnistClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)


# Load model
model = FCMnistClassifier()
checkpoint = torch.load('models/mnist_fc_classifier.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded!")
print(f"Test accuracy: {checkpoint['test_accuracy']:.2f}%")
print(f"Architecture: {checkpoint['architecture']}")

## 3. Load Test Data

In [ ]:
# Load MNIST test set
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Test samples: {len(test_dataset)}")

## 4. Select Test Image

In [ ]:
# Select a correctly classified test image
sample_idx = 0

# Find a correctly classified sample
for i in range(len(test_dataset)):
    img, label = test_dataset[i]
    with torch.no_grad():
        output = model(img.unsqueeze(0))
        pred = output.argmax(dim=1).item()
    if pred == label:
        sample_idx = i
        break

# Get the sample
test_image, true_label = test_dataset[sample_idx]
test_image_np = test_image.squeeze().numpy()

# Get prediction
with torch.no_grad():
    output = model(test_image.unsqueeze(0))
    pred_label = output.argmax(dim=1).item()
    pred_scores = torch.softmax(output, dim=1).squeeze().numpy()

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.imshow(test_image_np, cmap='gray')
ax1.set_title(f"Test Image (Label: {true_label})")
ax1.axis('off')

ax2.bar(range(10), pred_scores)
ax2.set_xlabel('Digit Class')
ax2.set_ylabel('Confidence')
ax2.set_title(f"Prediction: {pred_label} (Confidence: {pred_scores[pred_label]:.3f})")
ax2.set_xticks(range(10))

plt.tight_layout()
plt.show()

print(f"\nSelected image {sample_idx}: True label = {true_label}, Predicted = {pred_label}")

## 5. Define Input Perturbation Region

We create a Star set representing all images within ε (L∞ norm) of the test image.

In [ ]:
# Perturbation magnitude (L-infinity norm)
epsilon = 0.02  # 2% perturbation

# Flatten image to vector
img_vector = test_image.flatten().numpy().reshape(-1, 1)

# Create bounds: [img - ε, img + ε] clipped to [0, 1]
lb = np.clip(img_vector - epsilon, 0, 1)
ub = np.clip(img_vector + epsilon, 0, 1)

# Create Star set from bounds
input_star = Star.from_bounds(lb, ub)

print(f"Input Star Set:")
print(f"  Dimension: {input_star.dim}")
print(f"  Variables: {input_star.nVar}")
print(f"  Perturbation: ε = {epsilon}")
print(f"  Represents: All images within L∞ ball of radius {epsilon}")

## 6. Visualize Perturbation Region

In [ ]:
# Show lower and upper bounds
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(lb.reshape(28, 28), cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Lower Bound (img - ε)')
axes[0].axis('off')

axes[1].imshow(test_image_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Original Image')
axes[1].axis('off')

axes[2].imshow(ub.reshape(28, 28), cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Upper Bound (img + ε)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Any image in the perturbation region satisfies:")
print(f"  lb[i] ≤ perturbed_img[i] ≤ ub[i] for all pixels i")

## 7. Perform Reachability Analysis

Compute the **exact** output reachable set using Star sets.

In [ ]:
print("Starting reachability analysis...")
print("This may take a few minutes due to ReLU splitting.\n")

start_time = time.time()

# Compute reachable set
output_stars = reach_star_exact(model, [input_star])

elapsed_time = time.time() - start_time

print(f"\nReachability analysis complete!")
print(f"  Time: {elapsed_time:.2f} seconds")
print(f"  Input stars: 1")
print(f"  Output stars: {len(output_stars)}")
print(f"  Star splitting: {len(output_stars)}x (from ReLU layers)")

## 8. Compute Output Bounds

For each output star, compute the range of possible output values for each class.

In [ ]:
print("Computing output bounds for each star...\n")

# Compute bounds for each star
all_lbs = []
all_ubs = []

for i, star in enumerate(output_stars):
    star.estimate_ranges()
    all_lbs.append(star.state_lb.flatten())
    all_ubs.append(star.state_ub.flatten())
    print(f"Star {i+1}/{len(output_stars)}: lb shape = {star.state_lb.shape}, ub shape = {star.state_ub.shape}")

# Get overall bounds (union of all stars)
all_lbs = np.array(all_lbs)
all_ubs = np.array(all_ubs)

overall_lb = np.min(all_lbs, axis=0)
overall_ub = np.max(all_ubs, axis=0)

print(f"\nOverall output bounds:")
for i in range(10):
    print(f"  Class {i}: [{overall_lb[i]:.3f}, {overall_ub[i]:.3f}]")

## 9. Visualize Output Reachable Set

In [ ]:
# Plot output bounds
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(10)
width = 0.35

# Plot bounds
ax.bar(x - width/2, overall_lb, width, label='Lower Bound', alpha=0.7)
ax.bar(x + width/2, overall_ub, width, label='Upper Bound', alpha=0.7)

# Highlight true class
ax.axvline(true_label, color='green', linestyle='--', linewidth=2, label=f'True Class ({true_label})')

ax.set_xlabel('Output Class')
ax.set_ylabel('Network Output Value')
ax.set_title('Output Reachable Set (All Perturbed Inputs)')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Verify Local Robustness

Check if the true class has the highest output for **all** perturbations.

In [ ]:
print("=" * 60)
print("LOCAL ROBUSTNESS VERIFICATION")
print("=" * 60)

# Check robustness: true class lower bound > all other classes upper bounds
true_class_lb = overall_lb[true_label]
robust = True
violations = []

for i in range(10):
    if i != true_label:
        if overall_ub[i] >= true_class_lb:
            robust = False
            margin = overall_ub[i] - true_class_lb
            violations.append((i, margin))

print(f"\nTest Image: {sample_idx} (Label: {true_label})")
print(f"Perturbation: ε = {epsilon} (L∞ norm)")
print(f"\nTrue class ({true_label}) output range: [{overall_lb[true_label]:.3f}, {overall_ub[true_label]:.3f}]")
print()

if robust:
    print("✅ VERIFIED ROBUST")
    print(f"   The network correctly classifies ALL images within ε={epsilon}")
    print(f"   No adversarial examples exist in this region.")
else:
    print("❌ NOT ROBUST")
    print(f"   Potential adversarial examples exist!")
    print(f"\n   Classes that could beat true class {true_label}:")
    for cls, margin in violations:
        print(f"     Class {cls}: upper bound = {overall_ub[cls]:.3f} (margin: +{margin:.3f})")

print("\n" + "=" * 60)

## 11. Robustness Analysis

In [ ]:
# Compute safety margin for each class
safety_margins = true_class_lb - overall_ub
safety_margins[true_label] = np.inf  # Ignore self

# Plot safety margins
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['green' if m > 0 else 'red' for m in safety_margins]
colors[true_label] = 'blue'  # True class

bars = ax.bar(range(10), safety_margins, color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=1, linestyle='--')

ax.set_xlabel('Class')
ax.set_ylabel('Safety Margin')
ax.set_title('Robustness Safety Margins\n(Positive = Safe, Negative = Vulnerable)')
ax.set_xticks(range(10))
ax.grid(True, alpha=0.3, axis='y')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Safe (margin > 0)'),
    Patch(facecolor='red', alpha=0.7, label='Vulnerable (margin < 0)'),
    Patch(facecolor='blue', alpha=0.7, label='True Class')
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

print("\nSafety margins (true_class_lb - other_class_ub):")
for i in range(10):
    if i != true_label:
        status = "✅ Safe" if safety_margins[i] > 0 else "❌ Vulnerable"
        print(f"  Class {i}: {safety_margins[i]:+.3f}  {status}")

## 12. Summary Statistics

In [ ]:
print("\n" + "=" * 60)
print("VERIFICATION SUMMARY")
print("=" * 60)
print(f"\nModel: FC MNIST Classifier")
print(f"Architecture: 784 → 128 → 64 → 10")
print(f"\nTest Image: {sample_idx}")
print(f"True Label: {true_label}")
print(f"Predicted Label: {pred_label}")
print(f"\nVerification Settings:")
print(f"  Perturbation: ε = {epsilon} (L∞)")
print(f"  Method: Exact Star reachability")
print(f"\nResults:")
print(f"  Computation time: {elapsed_time:.2f}s")
print(f"  Output stars: {len(output_stars)}")
print(f"  Robustness: {'✅ VERIFIED ROBUST' if robust else '❌ NOT ROBUST'}")
if not robust:
    print(f"  Vulnerable to: {len(violations)} class(es)")
print("\n" + "=" * 60)

## 13. Try Different Epsilon Values

In [ ]:
# Test multiple epsilon values
epsilon_values = [0.01, 0.02, 0.03, 0.05]
robustness_results = []

print("Testing robustness at different perturbation levels...\n")

for eps in epsilon_values:
    # Create input star
    lb_eps = np.clip(img_vector - eps, 0, 1)
    ub_eps = np.clip(img_vector + eps, 0, 1)
    input_star_eps = Star.from_bounds(lb_eps, ub_eps)

    # Verify
    start = time.time()
    output_stars_eps = reach_star_exact(model, [input_star_eps])
    verify_time = time.time() - start

    # Check robustness
    all_lbs_eps = []
    all_ubs_eps = []
    for star in output_stars_eps:
        star.estimate_ranges()
        all_lbs_eps.append(star.state_lb.flatten())
        all_ubs_eps.append(star.state_ub.flatten())

    overall_lb_eps = np.min(all_lbs_eps, axis=0)
    overall_ub_eps = np.max(all_ubs_eps, axis=0)

    robust_eps = all(overall_lb_eps[true_label] > overall_ub_eps[i]
                     for i in range(10) if i != true_label)

    robustness_results.append({
        'epsilon': eps,
        'robust': robust_eps,
        'num_stars': len(output_stars_eps),
        'time': verify_time
    })

    status = "✅ ROBUST" if robust_eps else "❌ NOT ROBUST"
    print(f"ε = {eps:.3f}: {status:15s} ({len(output_stars_eps):4d} stars, {verify_time:.1f}s)")

print("\nConclusion:")
max_robust_eps = max([r['epsilon'] for r in robustness_results if r['robust']], default=0)
if max_robust_eps > 0:
    print(f"  Network is verified robust up to ε = {max_robust_eps:.3f}")
else:
    print(f"  Network is not robust at any tested epsilon value")

## Summary

✅ Loaded trained FC MNIST classifier  
✅ Performed exact reachability analysis with Star sets  
✅ Computed output bounds for perturbed inputs  
✅ Verified local robustness (or found vulnerability)  
✅ Analyzed safety margins for all classes

**Key Insights**:
- Star sets provide **exact** verification (sound and complete)
- ReLU layers cause star splitting (exponential growth)
- Smaller ε → easier to verify robust
- Safety margins show vulnerability to specific classes

**Next**: Train and verify a CNN in `03_train_cnn_mnist.ipynb` and `04_verify_cnn_mnist.ipynb`